# Semantic correspondence: Colab runner

This notebook only sets options and calls the repo code (`scripts/colab_setup.py`, `run.py`).
Code changes belong in the repo: edits made here are lost. Full guide: `RUN.md`.

**Before the first run:**
1. Runtime > Change runtime type > **GPU** (needed for `train`; `evaluate`/`submit` also work on CPU, slowly).
2. Accept the competition rules on Kaggle.
3. Add these to Colab Secrets (key icon) and allow notebook access: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `WANDB_API_KEY`, `HF_TOKEN`.

In [ ]:
# for charge the .env file as the project .env.example required
import os
from dotenv import load_dotenv

# Path to the .env file (supports both /content/.env and repository root .env)
env_path = '/content/.env' if os.path.exists('/content/.env') else '.env'

# Load the environment variables
if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path, override=True)
    print(f"Loaded environment variables from {env_path}")
else:
    print(f"Warning: {env_path} not found. Please make sure the file exists.")

In [ ]:
# Run settings
BRANCH = "giuseppe"      # branch to run; push your changes there first
MODE = "train"           # "train" | "evaluate" | "submit"   (settings: BASE in src/config.py)
RUN = ""                 # evaluate/submit an existing run (uses its best.pt)
RESUME = ""              # train: continue this run id after a disconnect
EXTRA = ""               # other run.py flags, e.g. "--dry-run"

# Transformation switches (override BASE in src/config.py for this Colab session):
CROSS_TRANSFORM = False     # File 1: src/models/cross_transform.py (multi-scale + cross-image attention adapter)
CONSTELLATION_ALIGN = False # File 2: src/models/constellation.py (pair-level Sim(2) constellation prior at inference)
OBJECT_ZOOM = True          # File 3: src/models/zoom.py (2-stage object-centric zoom at inference)
FUSION = "none"             # "none" (fast pure DINOv2) | "concat" | "adapter"

In [ ]:
# Clone the repo the first time, then let scripts/colab_setup.py do the rest:
# pull, install environment.yml (pip part), secrets, Drive, data and weights.
import os

REPO_URL = "https://github.com/julpfi/sapienza_aml_semantic_correspondence.git"
REPO_DIR = "/content/sapienza_aml_semantic_correspondence"
if not os.path.isdir(f"{REPO_DIR}/.git"):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%run scripts/colab_setup.py --branch {BRANCH}

In [ ]:
# Apply the transformation switches above to src/config.py before running run.py
import re
from pathlib import Path

cfg_path = Path("src/config.py")
text = cfg_path.read_text()
text = re.sub(r'("cross_transform":\s*)(True|False)', rf'\g<1>{CROSS_TRANSFORM}', text)
text = re.sub(r'("constellation_align":\s*)(True|False)', rf'\g<1>{CONSTELLATION_ALIGN}', text)
text = re.sub(r'("object_zoom":\s*)(True|False)', rf'\g<1>{OBJECT_ZOOM}', text)
text = re.sub(r'("fusion":\s*)"[^"]+"', rf'\g<1>"{FUSION}"', text)
cfg_path.write_text(text)
!grep -nE '"cross_transform":|"constellation_align":|"object_zoom":|"fusion":' src/config.py

In [ ]:
# Start the run. Runs are saved on your Drive: aml_semantic_correspondence/outputs/<run_id>
flags = f"--mode {MODE} {EXTRA}"
if RESUME:
    flags += f" --resume {RESUME}"
elif RUN:
    flags += f" --run {RUN}"
if MODE == "train":
    flags += " --require-gpu"
else:
    flags += " --object-zoom" if OBJECT_ZOOM else " --no-object-zoom"
    flags += " --constellation-align" if CONSTELLATION_ALIGN else " --no-constellation-align"
!python run.py {flags}

In [ ]:
# Optional: paired comparison of evaluated runs (seeds comma-separated, A then B), e.g. "a_s0,a_s1 b_s0,b_s1".
# Each run needs per_query.csv: MODE = "evaluate" with RUN = "<run_id>" writes it.
COMPARE = ""
if COMPARE:
    !python scripts/compare_runs.py {COMPARE}

In [ ]:
# Optional: upload a submission to Kaggle (set the run id printed above).
SUBMIT_RUN = "v1_confirmation"  # set to the run id you want to submit, or "" to skip
if SUBMIT_RUN:
    !python scripts/kaggle_io.py submit --file /content/drive/MyDrive/aml_semantic_correspondence/outputs/{SUBMIT_RUN}/submission.csv --message "{SUBMIT_RUN}"